## **Step 1: Download RAVDESS Dataset from Kaggle**

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ravdess-emotional-speech-audio' dataset.
Path to dataset files: /kaggle/input/ravdess-emotional-speech-audio


## **Step 2: Import Required Libraries**

In [5]:
import os
import numpy as np
import librosa

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout


## **Step 3: Emotion label mapping (RAVDESS standard)**

In [6]:
emotion_map = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}


## **Step 4: MFCC feature extraction function**
This converts audio into numbers the model can learn from.

In [7]:
def extract_mfcc(file_path):
    audio, sample_rate = librosa.load(file_path, duration=3, offset=0.5)
    mfcc = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
    mfcc = np.mean(mfcc.T, axis=0)
    return mfcc


## **Step 5: Load audio files and labels**

RAVDESS files are inside Actor folders.

In [8]:
X = []
y = []

for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".wav"):
            emotion_code = file.split("-")[2]
            emotion_label = emotion_map[emotion_code]

            file_path = os.path.join(root, file)
            mfcc_features = extract_mfcc(file_path)

            X.append(mfcc_features)
            y.append(emotion_label)


## **Step 6: Convert data to arrays**

In [9]:
X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)


(2880, 40)
(2880,)


## **Step 7: Encode emotion labels**

Convert text labels into numbers.

In [10]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_encoded = to_categorical(y_encoded)


## **Step 8: Train test split**

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42
)


## **Step 9: Reshape input for CNN**

CNN expects 4D input.

In [12]:
X_train = X_train.reshape(X_train.shape[0], 40, 1, 1)
X_test = X_test.reshape(X_test.shape[0], 40, 1, 1)


## **Step 10: Build CNN model**

In [13]:
model = Sequential()

model.add(Conv2D(32, (3,1), activation="relu", input_shape=(40,1,1)))
model.add(MaxPooling2D(pool_size=(2,1)))
model.add(Dropout(0.3))

model.add(Conv2D(64, (3,1), activation="relu"))
model.add(MaxPooling2D(pool_size=(2,1)))
model.add(Dropout(0.3))

model.add(Flatten())
model.add(Dense(128, activation="relu"))
model.add(Dense(y_train.shape[1], activation="softmax"))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## Step 11: Compile the model

In [14]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


## **Step 12: Train the model**

In [15]:
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_test, y_test)
)


Epoch 1/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.1441 - loss: 6.3657 - val_accuracy: 0.2552 - val_loss: 1.9339
Epoch 2/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2195 - loss: 2.2138 - val_accuracy: 0.2726 - val_loss: 1.8980
Epoch 3/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2557 - loss: 1.9952 - val_accuracy: 0.2899 - val_loss: 1.8548
Epoch 4/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2802 - loss: 1.8815 - val_accuracy: 0.3194 - val_loss: 1.8412
Epoch 5/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2893 - loss: 1.8585 - val_accuracy: 0.3646 - val_loss: 1.7623
Epoch 6/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3155 - loss: 1.7866 - val_accuracy: 0.3906 - val_loss: 1.7247
Epoch 7/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3400 - loss: 1.7587 - val_accuracy: 0.3854 - val_loss: 1.7161
Epoch 8/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3510 - loss: 1.7268 - val_accuracy: 0.4028 - val_loss

In [16]:
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_test, y_test)
)


Epoch 1/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6843 - loss: 0.8841 - val_accuracy: 0.7222 - val_loss: 0.8264
Epoch 2/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7028 - loss: 0.8357 - val_accuracy: 0.7639 - val_loss: 0.7563
Epoch 3/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7295 - loss: 0.7608 - val_accuracy: 0.7622 - val_loss: 0.7396
Epoch 4/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7192 - loss: 0.7674 - val_accuracy: 0.7656 - val_loss: 0.7149
Epoch 5/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7386 - loss: 0.7476 - val_accuracy: 0.7899 - val_loss: 0.6920
Epoch 6/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7550 - loss: 0.7004 - val_accuracy: 0.7812 - val_loss: 0.6879
Epoch 7/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7502 - loss: 0.6772 - val_accuracy: 0.7899 - val_loss: 0.6727
Epoch 8/30
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7759 - loss: 0.6532 - val_accuracy: 0.7934 - val_loss:

## **Step 11: Evaluate model**

In [17]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)


18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8882 - loss: 0.4405
Test Accuracy: 0.8940972089767456


## **Step 12: Save Model**

In [19]:
model.save("emotion_recognition_model.h5")


## **Step 13: Upload audio File For Prediction**

In [31]:


import numpy as np
import librosa
import os
import time

from tensorflow.keras.models import load_model
from sklearn.preprocessing import LabelEncoder
from google.colab import files

# ===== LOAD MODEL ONCE =====
MODEL_PATH = "/content/emotion_recognition_model.h5"

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError("Model not found at " + MODEL_PATH)

model = load_model(MODEL_PATH)
print("Model loaded successfully")

# ===== LABEL ORDER (training matched) =====
labels = [
    "angry",
    "calm",
    "disgust",
    "fearful",
    "happy",
    "neutral",
    "sad",
    "surprised"
]

label_encoder = LabelEncoder()
label_encoder.fit(labels)

# ===== MFCC FUNCTION =====
def extract_mfcc(file_path):
    audio, sr = librosa.load(file_path, sr=None, duration=3, offset=0.5)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    mfcc = np.mean(mfcc.T, axis=0)
    return mfcc

# ===== LOOP FOR MULTIPLE UPLOADS =====
while True:
    print("\nUpload an audio file (wav or mp3)")
    uploaded_audio = files.upload()

    if len(uploaded_audio) == 0:
        print("No file uploaded. Stopping.")
        break

    audio_file = list(uploaded_audio.keys())[0]

    while not os.path.exists(audio_file):
        time.sleep(0.2)

    mfcc = extract_mfcc(audio_file)
    mfcc = mfcc.reshape(1, 40, 1, 1)

    prediction = model.predict(mfcc)
    emotion = label_encoder.inverse_transform([np.argmax(prediction)])

    print("Predicted Emotion:", emotion[0])

    choice = input("Upload another audio? (y/n): ")
    if choice.lower() != "y":
        print("Session ended.")
        break


Model loaded successfully

Upload an audio file (wav or mp3)


Saving rb0m3zuowhc-surprise-sfx-0.mp3 to rb0m3zuowhc-surprise-sfx-0 (7).mp3


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 350ms/step
Predicted Emotion: angry
Upload another audio? (y/n): y

Upload an audio file (wav or mp3)


Saving 03-01-01-01-01-01-01.wav to 03-01-01-01-01-01-01 (3).wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Predicted Emotion: calm
Upload another audio? (y/n): y

Upload an audio file (wav or mp3)


Saving rb0m3zuowhc-surprise-sfx-0.mp3 to rb0m3zuowhc-surprise-sfx-0 (8).mp3
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Predicted Emotion: angry
Upload another audio? (y/n): n
Session ended.
